# DEXI Drone LLM Fine-Tuning

Fine-tune Qwen2.5-1.5B-Instruct for reliable drone tool-calling using QLoRA + Unsloth.

**Requirements:**
- Google Colab with **T4 GPU** (Runtime → Change runtime type → T4 GPU)
- Upload 5 files (see below)

Training takes ~20-30 minutes on T4. No API keys required.

## 1. Upload Files

Two uploads:
1. From `training/dataset/`: **train.jsonl**, **val.jsonl**
2. From `dexi_llm/config/`: **tools.json**, **system_prompt.txt**, **models.json**

In [ ]:
from google.colab import files

print("Upload #1: select train.jsonl and val.jsonl from training/dataset/")
uploaded = files.upload()
print(f"Got: {list(uploaded.keys())}\n")

print("Upload #2: select tools.json, system_prompt.txt, and models.json from dexi_llm/config/")
uploaded = files.upload()
print(f"Got: {list(uploaded.keys())}")

## 2. Install Dependencies

In [ ]:
!pip install unsloth

## 3. Load Config and Base Model

4-bit quantization — uses ~2GB VRAM.

In [ ]:
import json

MODEL_NAME = "qwen2.5-1.5b"

with open("models.json") as f:
    MODEL_CONFIG = json.load(f)[MODEL_NAME]

tpl = MODEL_CONFIG["chat_template"]

print(f"Model: {MODEL_CONFIG['base_model']}")
print(f"LoRA r={MODEL_CONFIG['lora']['r']}, alpha={MODEL_CONFIG['lora']['lora_alpha']}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_CONFIG["base_model"],
    max_seq_length=2048,
    load_in_4bit=True,
)

print(f"Loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

## 4. Add LoRA Adapters

In [ ]:
lora_cfg = MODEL_CONFIG["lora"]

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    target_modules=lora_cfg["target_modules"],
    lora_dropout=lora_cfg["lora_dropout"],
    bias="none",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable / 1e6:.1f}M / {total / 1e6:.0f}M ({100 * trainable / total:.1f}%)")

## 5. Build System Block

Load the shared config files (same ones used at runtime) and build the system prompt + tool definitions.

In [ ]:
with open("tools.json") as f:
    TOOLS = json.load(f)
with open("system_prompt.txt") as f:
    SYSTEM_PROMPT = f.read().strip()

tools_json = "\n".join(
    json.dumps(t, separators=(",", ":")) for t in TOOLS
)
SYSTEM_BLOCK = (
    f"{SYSTEM_PROMPT}\n\n"
    "# Tools\n\n"
    "You may call one or more functions to assist with the user query.\n\n"
    "You are provided with function signatures within <tools></tools> XML tags:\n"
    f"<tools>\n{tools_json}\n</tools>\n\n"
    "For each function call, return a json object with function name and "
    "arguments within <tool_call></tool_call> XML tags:\n"
    "<tool_call>\n"
    '{"name": <function-name>, "arguments": <args-json-object>}\n'
    "</tool_call>"
)

print(f"System block: {len(SYSTEM_BLOCK)} chars")

## 6. Train

3 epochs, effective batch size 16 (2 × 8 gradient accumulation). ~20-30 min on T4.

A custom collator masks loss on prompt tokens so the model only trains on assistant responses
(same effect as the removed `DataCollatorForCompletionOnlyLM`).

In [ ]:
import os
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

os.environ["WANDB_DISABLED"] = "true"

dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "val": "val.jsonl",
})


def format_example(example):
    """Format a single example as ChatML text with system block injected."""
    text = tpl["bos"]
    text += tpl["system_prefix"] + SYSTEM_BLOCK + tpl["system_suffix"]
    for msg in example["messages"]:
        role = msg["role"]
        if role == "user":
            text += tpl["user_prefix"] + msg["content"] + tpl["user_suffix"]
        elif role == "assistant":
            text += tpl["assistant_prefix"] + msg["content"] + tpl["assistant_suffix"]
    return {"text": text}


dataset = dataset.map(format_example)
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['val'])}")
print(f"Sample (first 300 chars): {dataset['train'][0]['text'][:300]}")


# Trivial formatting_func — Unsloth requires this, but text is pre-computed.
# Handles both single-example calls (returns str) and batched calls (returns list).
def formatting_func(example):
    return example["text"]


# Completion-only collator: masks loss on system+user tokens,
# trains only on assistant responses.
response_template_ids = tokenizer.encode(
    tpl["assistant_prefix"], add_special_tokens=False
)
pad_id = (
    tokenizer.pad_token_id
    if tokenizer.pad_token_id is not None
    else tokenizer.eos_token_id
)


class CompletionOnlyCollator:
    def __call__(self, features):
        input_ids_list = [f["input_ids"] for f in features]
        max_len = max(len(ids) for ids in input_ids_list)
        tpl_len = len(response_template_ids)

        batch_ids, batch_mask, batch_labels = [], [], []
        for ids in input_ids_list:
            pad_len = max_len - len(ids)
            labels = list(ids) + [-100] * pad_len

            # Find the assistant prefix and mask everything before it
            for i in range(len(ids) - tpl_len + 1):
                if ids[i : i + tpl_len] == response_template_ids:
                    labels[: i + tpl_len] = [-100] * (i + tpl_len)

            batch_ids.append(ids + [pad_id] * pad_len)
            batch_mask.append([1] * len(ids) + [0] * pad_len)
            batch_labels.append(labels)

        return {
            "input_ids": torch.tensor(batch_ids),
            "attention_mask": torch.tensor(batch_mask),
            "labels": torch.tensor(batch_labels),
        }


# Auto-detect model precision
use_bf16 = model.get_input_embeddings().weight.dtype == torch.bfloat16

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    formatting_func=formatting_func,
    data_collator=CompletionOnlyCollator(),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=10,
        eval_strategy="epoch",
        output_dir=MODEL_CONFIG["training"]["lora_dir"],
        seed=42,
        report_to="none",
        max_seq_length=2048,
    ),
)

print(f"Training ({'bf16' if use_bf16 else 'fp16'})...")
results = trainer.train()
print(f"\nDone! Train loss: {results.training_loss:.4f}")

## 7. Test

8 prompts — 7 should produce `<tool_call>`, 1 should politely refuse.

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    # (prompt, expect_tool_call)
    ("make the led green", True),
    ("fly forward 2 meters", True),
    ("what's my battery level?", True),
    ("do a flip", True),
    ("set servo 3 to 90 degrees", True),
    ("what topics are available?", True),
    ("start recording video", True),
    ("play some music", False),  # should refuse
]


def build_prompt(user_msg):
    """Build prompt using the same template as training."""
    messages = [
        {"role": "system", "content": SYSTEM_BLOCK},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


print(f"{'='*70}")
print(f"  TEST RESULTS: {MODEL_NAME}")
print(f"{'='*70}")

passed = 0

for prompt, expect_tool in test_cases:
    input_text = build_prompt(prompt)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    has_tool = "<tool_call>" in response
    correct = has_tool == expect_tool
    if correct:
        passed += 1
    status = "PASS" if correct else "FAIL"
    expect_str = "tool_call" if expect_tool else "refusal"
    got_str = "tool_call" if has_tool else "text"

    print(f"\n[{status}] \"{prompt}\"")
    print(f"  Expected: {expect_str} | Got: {got_str}")
    print(f"  Response: {response[:200]}")

print(f"\n{'='*70}")
print(f"  Score: {passed}/{len(test_cases)} passed")
print(f"{'='*70}")

## 8. Export to GGUF

Merge LoRA weights and export as Q4_K_M GGUF (~1GB) for llama.cpp deployment.

In [ ]:
import glob

merged_dir = MODEL_CONFIG["training"]["merged_dir"]
gguf_dir = MODEL_CONFIG["training"]["gguf_dir"]
gguf_name = MODEL_CONFIG["gguf_name"]

print(f"Merging LoRA weights to {merged_dir}...")
model.save_pretrained_merged(merged_dir, tokenizer)

print(f"\nExporting to GGUF (Q4_K_M) in {gguf_dir}...")
model.save_pretrained_gguf(
    gguf_dir,
    tokenizer,
    quantization_method="q4_k_m",
)

# Find the GGUF — Unsloth appends "_gguf" to the output directory name
# Search both the requested dir and the Unsloth-created one
search_dirs = [gguf_dir, f"{gguf_dir}_gguf"]
gguf_files = []
for d in search_dirs:
    found = glob.glob(f"{d}/*.gguf")
    if found:
        print(f"\nFound .gguf files in {d}/:")
        for f in found:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"  {os.path.basename(f)} ({size_mb:.0f} MB)")
        gguf_files.extend(found)

# Pick the largest .gguf (the quantized model, not vocab/bf16 intermediates)
gguf_files = sorted(gguf_files, key=os.path.getsize, reverse=True)

if gguf_files:
    gguf_path = gguf_files[0]
    size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
    print(f"\nReady to download: {gguf_path} ({size_mb:.0f} MB)")
else:
    print(f"\nERROR: No .gguf files found in {search_dirs}")
    for d in search_dirs:
        if os.path.isdir(d):
            print(f"\nContents of {d}/:")
            !ls -la {d}/
    # Last resort: search everywhere under /content
    print("\nSearching /content for any .gguf files:")
    !find /content -name "*.gguf" -ls 2>/dev/null || echo "  None found"

## 9. Download

Download the GGUF, then deploy:

```bash
# Copy into models dir and rename
cp ~/Downloads/unsloth.Q4_K_M.gguf dexi_ws/src/dexi_llm/models/dexi-qwen2.5-1.5b-q4_k_m.gguf

# Or SCP to Pi
scp ~/Downloads/unsloth.Q4_K_M.gguf dexi@192.168.68.59:~/dexi_ws/src/dexi_llm/models/dexi-qwen2.5-1.5b-q4_k_m.gguf

# Launch
ros2 launch dexi_llm llm_node.launch.py backend:=llama_cpp \
    model_path:=<path>/dexi-qwen2.5-1.5b-q4_k_m.gguf
```

In [ ]:
from google.colab import files

try:
    gguf_path
except NameError:
    print("ERROR: gguf_path not set. Re-run the export cell above.")
else:
    if not os.path.exists(gguf_path):
        print(f"ERROR: {gguf_path} not found. Re-run the export cell above.")
    else:
        size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
        print(f"Downloading: {gguf_path} ({size_mb:.0f} MB)")
        files.download(gguf_path)